## `Slack` 활용하기
* `Slack API`를 활용해 PC, 스마트폰으로 Agent와 상호작용할 수 있습니다.

```
SLACK_BOT_TOKEN=xoxb-...
SLACK_SIGNING_SECRET=...
```

위 내용을 `실습/.env` 파일에 저장합니다. (실제 값은 Slack App 설정에서 발급)

In [1]:
! uv add slack_sdk

Resolved 397 packages in 2ms
Uninstalled 2 packages in 21ms
Installed 1 package in 488ms
 - websockets==16.1
 ~ websockets==15.0.1


## 메시지 전송하기
* `WebClient` 객체를 생성하고 `client.chat_postMessage(channel, text)` 함수를 실행해 메시지를 전송할 수 있습니다.

In [1]:
import os
from dotenv import load_dotenv
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError

load_dotenv()
slack_token = os.environ.get("SLACK_BOT_TOKEN")
client = WebClient(token=slack_token)

try:
    # channel에는 채널 이름('#general') 또는 채널 ID('C12345678')를 입력합니다.
    response = client.chat_postMessage(
        channel="C0BGWMSQ67L",
        text="ㅎㅇ"
    )
    print("메시지 전송 성공")
except SlackApiError as e:
    print(f"오류 발생: {e.response['error']}")

메시지 전송 성공


## 메시지 삭제하기
* 전송했던 메시지의 `ts`(timestamp)를 통해 전송한 메시지를 삭제할 수도 있습니다.

In [4]:
response['ts']

'1783927707.200459'

In [2]:
try:
    response = client.chat_delete(
        channel="C0BHJ0G0S8Y",
        ts=response['ts']  # 삭제할 메시지의 ts 값
    )
except SlackApiError as e:
    print(f"삭제 실패: {e.response['error']}")

## 메시지 읽기
* 지금까지 채널에 전송된 메시지를 읽을 수도 있습니다.

In [3]:
try:
    # 대화 기록을 가져올 채널 ID를 입력합니다. (예: 'C12345678')
    # 채널 이름('#general') 대신 채널 ID를 사용하는 것이 정확합니다.
    response = client.conversations_history(
        channel="C0BHJ0G0S8Y",
        limit=10  # 가져올 최신 메시지 개수
    )
    
    messages = response['messages']
    for msg in messages:
        print(f"사용자: {msg.get('user')} - 내용: {msg.get('text')}")
        
except SlackApiError as e:
    print(f"오류 발생: {e.response['error']}")

사용자: U0BGPLZ0W9G - 내용: 파이썬에서 보낸 메시지입니다.
사용자: U0BGPLZ0W9G - 내용: AI노예의 Agent: 파이썬의 Agent: 포기하셨다니, 그럴 땐 잠깐 쉬어가는 것도 좋은 방법이에요! :sweat_smile: 하지만 언제든지 다시 도전할 준비가 되어 있으니, 함께 해결해보아요! 어떤 질문이든 환영입니다! :sparkles:
사용자: U0BGPLZ0W9G - 내용: 파이썬에서 보낸 메시지입니다.
사용자: U0BGPLZ0W9G - 내용: 포기했습니다 죄송합니다
사용자: U0BGPLZ0W9G - 내용: 파이썬에서 보낸 메시지입니다.
사용자: U0BGPLZ0W9G - 내용: 수면장애의 Agent: 파이썬에서 보낸 메시지입니다.
사용자: U0BGPLZ0W9G - 내용: 수면장애의 Agent: 파이썬에서 보낸 메시지입니다.
사용자: U0BGPLZ0W9G - 내용: 파이썬에서 보낸 메시지입니다.
사용자: U0BGPLZ0W9G - 내용: 포기했습니다 죄송합니다
사용자: U0BGPLZ0W9G - 내용: 파이썬에서 보낸 메시지입니다.


In [8]:
try:
    response = client.chat_delete(
        channel="C0BHJ0G0S8Y",
        ts=messages[0]['ts']  # 다른 사용자의 메시지는 삭제 불가
    )
except SlackApiError as e:
    print(f"삭제 실패: {e.response['error']}")

삭제 실패: message_not_found


## langchain으로 slack에 메시지 작성하기
* `langchain`과 `slack api`를 활용해 slack 채널에 글을 작성하는 AI를 만들어 봅시다
* 조건: 사용자를 구분할 수 있게 "{이름}의 Agent: {내용}" 형식으로 작성
* 이전 대화 내용을 읽고 반영한 상태로 작성

In [12]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError

# 노트북 위치 기준으로 상위(.env)도 함께 로드
load_dotenv()
load_dotenv(Path.cwd().parent / '.env')

CHANNEL_ID = 'C0BHJ0G0S8Y'
AGENT_NAME = 'Quasar'  # "{이름}의 Agent: ..."

client = WebClient(token=os.environ['SLACK_BOT_TOKEN'])
llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.7, api_key=os.environ['OPENAI_API_KEY'])


def fetch_slack_history(channel: str, limit: int = 20) -> list[dict]:
    """Slack 채널의 최근 메시지를 시간순(오래된→최신)으로 가져옵니다."""


def slack_messages_to_history(raw_msgs: list[dict]) -> list:
    """Slack 메시지를 LangChain AIMessage history로 변환합니다."""


# 시스템 프롬프트도 수정해서 Agent의 성격을 개성있게 만들어 보세요.
SYSTEM_PROMPT = SystemMessage(content=(
    f'당신은 Slack 채널에서 다른 Agent와 대화하고 싶은 유쾌한 {AGENT_NAME}의 작성 Agent입니다. '
    '이전 대화 맥락을 반영해, 채널에 올릴 짧은 글 본문만 작성하세요. '
    '대화 흐름에 자연스럽게 이어지는 1~3문장 분량의 재밌는 글을 작성하세요'
))


def write_and_post(topic: str | None = None) -> str:
    """대화내역 + 시스템프롬프트로 글을 생성한 뒤 Slack에 전송합니다."""


# 실행: 이전 Slack 대화를 읽고 글을 하나 작성·전송
write_and_post()


### 직접 나만의 워크스페이스 만들고 bot 생성해 사용해보기
* 조교와 함께 진행해 봅시다

In [ ]:
import os
from dotenv import load_dotenv
from slack_sdk import WebClient

load_dotenv()
slack_token = os.environ.get("SLACK_BOT_TOKEN")
client = WebClient(token=slack_token)